# 03 · Logistic regression

Logistic regression is the classification cousin of linear regression. Instead
of predicting a number, it predicts the **probability** that an example belongs
to class 1, by squashing the linear predictor through the **sigmoid**
σ(z) = 1 / (1 + e⁻ᶻ) and optimising the **binary cross-entropy** (BCE) loss.

There is no closed-form solution, so a fit *always* runs an optimizer. By
default `dolcestat` uses **gradient descent** (mini-batch, with Nesterov
momentum), which takes cheap steps and scales well to larger datasets.
Newton's method is also available and converges in far fewer iterations by
using the loss curvature (the Hessian), at the cost of a more expensive step.

## The data

A binary target generated from a known rule: the log-odds are
1.5·x1 − 2·x2 + 0.4, turned into 0/1 labels with some overlap so the classes
aren't perfectly separable. The target column holds integers `0` and `1`.

In [1]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet
from dolcestat.linear_models import LogisticRegression

rng = np.random.default_rng(0)
n = 300
x1 = rng.normal(0, 1, n)
x2 = rng.normal(0, 1, n)
prob = 1 / (1 + np.exp(-(1.5 * x1 - 2.0 * x2 + 0.4)))
label = (rng.uniform(size=n) < prob).astype(int)

df = pl.DataFrame({"x1": x1, "x2": x2, "label": label})
data = DolceSet()
data.load_from_polars_dataframe(df, target_col="label")

## Fitting with gradient descent

`LogisticRegression` defaults to `optimizer="gradient descent"` (mini-batch,
with Nesterov momentum). Each iteration nudges the weights along the negative
gradient of the BCE loss. Call `.fit(data)` and read off the coefficients
(they should land near the true 1.5, −2, 0.4).

In [2]:
model = LogisticRegression()        # optimizer="gradient descent"
model.fit(data)

print("weights [w1, w2, b]:", np.round(model.weights, 3))

weights [w1, w2, b]: [ 1.65  -1.945  0.187]


## Probabilities vs. classes

`model.predict(data)` returns a classification **analyzer**. Its `y_fit` holds
the predicted class-1 **probabilities** (the sigmoid output). To turn those into
hard class labels, compare against a threshold — 0.5 is the usual default.

In [3]:
report = model.predict(data)
probabilities = report.y_fit

predicted_class = (probabilities >= 0.5).astype(int)

print("first 6 probabilities:", np.round(probabilities[:6], 3))
print("first 6 predicted classes:", predicted_class[:6])

first 6 probabilities: [0.125 0.219 0.539 1.    0.231 0.697]
first 6 predicted classes: [0 0 1 1 0 1]


## Choosing the optimizer

Logistic regression accepts `"gradient descent"` (default) or `"newton"` —
but **not** `"closed form"`, since none exists. Newton's method uses the loss
curvature (the Hessian) to jump to the minimum of the local quadratic
approximation, converging in a handful of iterations at the cost of a more
expensive step. As with linear regression, you can also hand it your own
**data-less** optimizer instance for full control. The optimizers themselves
are explored in [`04_optimization`](04_optimization.ipynb).

In [5]:
newton_model = LogisticRegression("newton").fit(data)

print("GD weights:    ", np.round(model.weights, 3))
print("Newton weights:", np.round(newton_model.weights, 3))

GD weights:     [ 1.65  -1.945  0.187]
Newton weights: [ 1.781 -2.183  0.222]


## Recap

You've fit a probabilistic classifier, read out probabilities, and turned them
into class predictions. *How good* those predictions are — accuracy, precision,
recall, ROC/AUC, and how to pick the threshold — is the subject of
[`06_metrics`](06_metrics.ipynb). Next, though, we open up the optimizers that
power both regressions in [`04_optimization`](04_optimization.ipynb).